# Aplicando os métodos de regressão

Carregamos os dados que já foram pré-processados e submetidos à Redução de Dimensionalidade (PCA). É fundamental separar as variáveis identificadoras (TEAM_ID e SEASON_ID) da matriz de features para garantir que os modelos não utilizem essas informações de forma indevida durante o treinamento (evitando vazamento de dados). Por fim, isolamos a nossa variável alvo: PLAYOFF_WINS.

In [3]:
import pandas as pd

# carregando os dados (Features reduzidas e Alvo)
X_train_full = pd.read_csv('./CSVs/X_train_pca.csv')
X_test_full = pd.read_csv('./CSVs/X_test_pca.csv')
y_train_df = pd.read_csv('./CSVs/y_train_final.csv')
y_test_df = pd.read_csv('./CSVs/y_test_final.csv')

# separando os identificadores das features (O modelo não pode ver os IDs)
colunas_id = ['TEAM_ID', 'SEASON_ID']

X_train = X_train_full.drop(columns=colunas_id)
X_test = X_test_full.drop(columns=colunas_id)

# convertendo o alvo para Pandas Series (formato exigido pelo Scikit-Learn)
y_train = y_train_df['PLAYOFF_WINS']
y_test = y_test_df['PLAYOFF_WINS']

# verificação 
print("--- Dados carregados e prontos para o Torneio de Modelos ---")
print(f"Treino: Matriz X = {X_train.shape} | Vetor y = {y_train.shape}")
print(f"Teste:  Matriz X = {X_test.shape}   | Vetor y = {y_test.shape}")
print(f"\nVariáveis preditoras ativas: {X_train.columns.tolist()}")

--- Dados carregados e prontos para o Torneio de Modelos ---
Treino: Matriz X = (802, 5) | Vetor y = (802,)
Teste:  Matriz X = (30, 5)   | Vetor y = (30,)

Variáveis preditoras ativas: ['PC1', 'PC2', 'PC3', 'PC4', 'PC5']


Para garantir uma comparação justa e direta entre todas as técnicas de regressão que testaremos, criamos uma função auxiliar unificada. Ela calcula as principais métricas de erro (MAE, MSE e RMSE) e a métrica de ajuste (R²). Essa função também nos ajudará a estruturar um quadro comparativo ao final de todas as execuções.

In [4]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# criando a Função Padrão de Avaliação (Para usar em todos os modelos)
def avaliar_modelo(nome_modelo, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    
    print(f"--- Resultados: {nome_modelo} ---")
    print(f"MAE  (Erro médio absoluto):      {mae:.4f}")
    print(f"MSE  (Penaliza erros grandes):   {mse:.4f}")
    print(f"RMSE (Mesma unidade de y):       {rmse:.4f}")
    print(f"R²   (Ganho sobre a média):      {r2:.4f}")
    
    # retorna um dicionário para podermos montar uma tabela comparativa no final
    return {'Modelo': nome_modelo, 'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}


Iniciamos o nosso torneio de modelos estabelecendo um baseline (linha de base) com a Regressão Linear. Este algoritmo tentará encontrar a melhor relação linear entre os nossos componentes principais (PCs) e o número de vitórias nos playoffs. É um modelo mais simples, rápido e que nos dará uma boa noção inicial do comportamento dos dados.

In [5]:
# ==========================================
# treinando e avaliando a Regressão Linear
# ==========================================

lr_pca = LinearRegression()
lr_pca.fit(X_train, y_train)

y_pred_lr = lr_pca.predict(X_test)

# chamando a nossa função de avaliação
metricas_lr = avaliar_modelo("Regressão Linear (PCA-5)", y_test, y_pred_lr)


--- Resultados: Regressão Linear (PCA-5) ---
MAE  (Erro médio absoluto):      2.6231
MSE  (Penaliza erros grandes):   10.9812
RMSE (Mesma unidade de y):       3.3138
R²   (Ganho sobre a média):      0.3759


Como isolamos os identificadores no início, criamos este dicionário auxiliar para traduzir os IDs numéricos da NBA de volta para os nomes reais das franquias. Isso facilitará a interpretação qualitativa dos resultados nas nossas tabelas de previsão.

In [6]:
# dicionário padrão de IDs da NBA para os nomes das franquias
# pra melhorar visualização posteriormente
nba_teams = {
    1610612737: 'Atlanta Hawks', 1610612738: 'Boston Celtics', 1610612739: 'Cleveland Cavaliers',
    1610612740: 'New Orleans Pelicans', 1610612741: 'Chicago Bulls', 1610612742: 'Dallas Mavericks',
    1610612743: 'Denver Nuggets', 1610612744: 'Golden State Warriors', 1610612745: 'Houston Rockets',
    1610612746: 'LA Clippers', 1610612747: 'Los Angeles Lakers', 1610612748: 'Miami Heat',
    1610612749: 'Milwaukee Bucks', 1610612750: 'Minnesota Timberwolves', 1610612751: 'Brooklyn Nets',
    1610612752: 'New York Knicks', 1610612753: 'Orlando Magic', 1610612754: 'Indiana Pacers',
    1610612755: 'Philadelphia 76ers', 1610612756: 'Phoenix Suns', 1610612757: 'Portland Trail Blazers',
    1610612758: 'Sacramento Kings', 1610612759: 'San Antonio Spurs', 1610612760: 'Oklahoma City Thunder',
    1610612761: 'Toronto Raptors', 1610612762: 'Utah Jazz', 1610612763: 'Memphis Grizzlies',
    1610612764: 'Washington Wizards', 1610612765: 'Detroit Pistons', 1610612766: 'Charlotte Hornets'
}

Abaixo, unimos as previsões geradas pelo modelo linear com o número real de vitórias na temporada de teste. Ordenar esses dados nos ajuda a identificar rapidamente quais times o modelo conseguiu prever com maior precisão e onde ele superestimou ou subestimou o desempenho.

In [7]:

# visualizando as previsões
print("\n--- Times (Ordenados por Vitórias Reais) ---")
resultados_lr = X_test_full[['TEAM_ID', 'SEASON_ID']].copy()
resultados_lr['VITORIAS_REAIS'] = y_test.values
resultados_lr['PREVISAO_LR'] = y_pred_lr.round(1)

# aplicando o mapeamento no DataFrame de resultados
resultados_lr['TEAM_NAME'] = resultados_lr['TEAM_ID'].map(nba_teams)

# reorganizando as colunas para o nome aparecer logo depois do ID e facilitar a leitura
colunas_ordem = ['TEAM_ID', 'TEAM_NAME', 'SEASON_ID', 'VITORIAS_REAIS', 'PREVISAO_LR']
resultados_lr = resultados_lr[colunas_ordem]

display(resultados_lr.sort_values(by='VITORIAS_REAIS', ascending=False))


--- Times (Ordenados por Vitórias Reais) ---


,TEAM_ID,TEAM_NAME,SEASON_ID,VITORIAS_REAIS,PREVISAO_LR
1,1610612738,Boston Celtics,2023-24,16,8.7
6,1610612742,Dallas Mavericks,2023-24,13,3.8
17,1610612750,Minnesota Timberwolves,2023-24,9,6.9
11,1610612754,Indiana Pacers,2023-24,8,3.9
19,1610612752,New York Knicks,2023-24,7,5.0
7,1610612743,Denver Nuggets,2023-24,7,6.1
20,1610612760,Oklahoma City Thunder,2023-24,6,6.5
5,1610612739,Cleveland Cavaliers,2023-24,5,4.6
21,1610612753,Orlando Magic,2023-24,3,4.6
16,1610612749,Milwaukee Bucks,2023-24,2,4.4


Nosso segundo modelo introduz não-linearidade. A Árvore de Decisão particiona o espaço das variáveis do PCA para fazer suas estimativas. Para evitar que o algoritmo simplesmente "decore" os dados de treino (overfitting) e perca a capacidade de generalização, limitamos o crescimento da árvore definindo o parâmetro de profundidade máxima (max_depth=5).

In [8]:
from sklearn.tree import DecisionTreeRegressor

# inicializando a Árvore de Decisão
# Usamos max_depth para evitar que a árvore cresça infinitamente e "decore" o treino (overfitting)
# O random_state garante que o resultado seja o mesmo toda vez que rodar
dt_pca = DecisionTreeRegressor(max_depth=5, random_state=42)

# treinando o Modelo
dt_pca.fit(X_train, y_train)

# fazendo as Previsões
y_pred_dt = dt_pca.predict(X_test)

# avaliando usando a nossa função
metricas_dt = avaliar_modelo("Decision Tree (PCA-5)", y_test, y_pred_dt)

--- Resultados: Decision Tree (PCA-5) ---
MAE  (Erro médio absoluto):      2.7360
MSE  (Penaliza erros grandes):   22.0040
RMSE (Mesma unidade de y):       4.6908
R²   (Ganho sobre a média):      -0.2505


Comparativo estruturado das previsões feitas pela Árvore de Decisão frente aos resultados reais da temporada.

In [9]:

# visualizando as previsões
print("\n--- Times (Ordenados por Vitórias Reais) ---")
resultados_dt = X_test_full[['TEAM_ID', 'SEASON_ID']].copy()
resultados_dt['VITORIAS_REAIS'] = y_test.values
resultados_dt['PREVISAO_DT'] = y_pred_dt.round(1)

# aplicando o mapeamento no DataFrame de resultados
resultados_dt['TEAM_NAME'] = resultados_dt['TEAM_ID'].map(nba_teams)

# reorganizando as colunas para o nome aparecer logo depois do ID e facilitar a leitura
colunas_ordem = ['TEAM_ID', 'TEAM_NAME', 'SEASON_ID', 'VITORIAS_REAIS', 'PREVISAO_DT']
resultados_dt = resultados_dt[colunas_ordem]

display(resultados_dt.sort_values(by='VITORIAS_REAIS', ascending=False))


--- Times (Ordenados por Vitórias Reais) ---


,TEAM_ID,TEAM_NAME,SEASON_ID,VITORIAS_REAIS,PREVISAO_DT
1,1610612738,Boston Celtics,2023-24,16,6.6
6,1610612742,Dallas Mavericks,2023-24,13,0.8
17,1610612750,Minnesota Timberwolves,2023-24,9,9.7
11,1610612754,Indiana Pacers,2023-24,8,4.5
19,1610612752,New York Knicks,2023-24,7,5.2
7,1610612743,Denver Nuggets,2023-24,7,10.5
20,1610612760,Oklahoma City Thunder,2023-24,6,10.5
5,1610612739,Cleveland Cavaliers,2023-24,5,5.2
21,1610612753,Orlando Magic,2023-24,3,3.5
16,1610612749,Milwaukee Bucks,2023-24,2,0.8


Para mitigar a alta variância característica de uma única Árvore de Decisão, evoluímos para um método ensemble: o Random Forest. Este modelo cria múltiplas árvores de decisão durante o treinamento (neste caso, 100 estimadores) e calcula a média das previsões de todas elas, entregando, geralmente, um resultado mais robusto e estável.

In [10]:
from sklearn.ensemble import RandomForestRegressor

# inicializando o Random Forest
# n_estimators: Número de árvores na floresta (100 é um bom padrão)
# max_depth: Profundidade máxima de cada árvore
# random_state: Para garantir reprodutibilidade
rf_pca = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)

# treinando o Modelo
rf_pca.fit(X_train, y_train)

# fazendo as Previsões
y_pred_rf = rf_pca.predict(X_test)

# avaliando usando a nossa função
metricas_rf = avaliar_modelo("Random Forest (PCA-5)", y_test, y_pred_rf)

--- Resultados: Random Forest (PCA-5) ---
MAE  (Erro médio absoluto):      2.3398
MSE  (Penaliza erros grandes):   12.6735
RMSE (Mesma unidade de y):       3.5600
R²   (Ganho sobre a média):      0.2797


Análise do desempenho preditivo do modelo Random Forest em relação às vitórias reais observadas nos playoffs.

In [11]:
# visualizando as previsões
print("\n--- Times (Ordenados por Vitórias Reais) ---")
resultados_rf = X_test_full[['TEAM_ID', 'SEASON_ID']].copy()
resultados_rf['TEAM_NAME'] = resultados_rf['TEAM_ID'].map(nba_teams)
resultados_rf['VITORIAS_REAIS'] = y_test.values
resultados_rf['PREVISAO_RF'] = y_pred_rf.round(1)

# reorganizando as colunas
colunas_ordem = ['TEAM_ID', 'TEAM_NAME', 'SEASON_ID', 'VITORIAS_REAIS', 'PREVISAO_RF']
resultados_rf = resultados_rf[colunas_ordem]

display(resultados_rf.sort_values(by='VITORIAS_REAIS', ascending=False))


--- Times (Ordenados por Vitórias Reais) ---


,TEAM_ID,TEAM_NAME,SEASON_ID,VITORIAS_REAIS,PREVISAO_RF
1,1610612738,Boston Celtics,2023-24,16,7.5
6,1610612742,Dallas Mavericks,2023-24,13,3.9
17,1610612750,Minnesota Timberwolves,2023-24,9,9.3
11,1610612754,Indiana Pacers,2023-24,8,5.4
19,1610612752,New York Knicks,2023-24,7,5.8
7,1610612743,Denver Nuggets,2023-24,7,9.1
20,1610612760,Oklahoma City Thunder,2023-24,6,10.0
5,1610612739,Cleveland Cavaliers,2023-24,5,4.9
21,1610612753,Orlando Magic,2023-24,3,2.7
16,1610612749,Milwaukee Bucks,2023-24,2,4.0


Enquanto o Random Forest treina dezenas de árvores de forma isolada e simultânea (e depois tira a média — uma técnica chamada Bagging), o XGBoost constrói as árvores em série. A árvore número 2 é construída focando exclusivamente nos erros que a árvore número 1 cometeu, a árvore 3 foca nos erros da 2, e assim por diante (uma técnica chamada Boosting).

In [16]:
from xgboost import XGBRegressor

# 1. Inicializar o XGBoost
# n_estimators: número de árvores sequenciais
# max_depth: profundidade de cada árvore (geralmente usa-se árvores mais "rasas" que no RF, como 3 ou 4)
# learning_rate: o "tamanho do passo" de correção entre uma árvore e outra (0.1 é um bom padrão)
xgb_pca = XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)

# Treinar o Modelo
xgb_pca.fit(X_train, y_train)

# Fazer Previsões
y_pred_xgb = xgb_pca.predict(X_test)

# Avaliar usando a nossa função padronizada
metricas_xgb = avaliar_modelo("XGBoost (PCA-5)", y_test, y_pred_xgb)

--- Resultados: XGBoost (PCA-5) ---
MAE  (Erro médio absoluto):      2.4673
MSE  (Penaliza erros grandes):   14.6340
RMSE (Mesma unidade de y):       3.8254
R²   (Ganho sobre a média):      0.1683


Visualização das previsões:

In [17]:
# Organizar e Visualizar os Resultados
resultados_xgb = X_test_full[['TEAM_ID', 'SEASON_ID']].copy()
resultados_xgb['TEAM_NAME'] = resultados_xgb['TEAM_ID'].map(nba_teams)
resultados_xgb['VITORIAS_REAIS'] = y_test.values
resultados_xgb['PREVISAO_XGB'] = y_pred_xgb.round(1)

# Reorganizar colunas
colunas_ordem = ['TEAM_ID', 'TEAM_NAME', 'SEASON_ID', 'VITORIAS_REAIS', 'PREVISAO_XGB']
resultados_xgb = resultados_xgb[colunas_ordem]

print("\n--- Ranking dos Times (XGBoost) ---")
display(resultados_xgb.sort_values(by='VITORIAS_REAIS', ascending=False))


--- Ranking dos Times (XGBoost) ---


,TEAM_ID,TEAM_NAME,SEASON_ID,VITORIAS_REAIS,PREVISAO_XGB
1,1610612738,Boston Celtics,2023-24,16,9.1
6,1610612742,Dallas Mavericks,2023-24,13,4.6
17,1610612750,Minnesota Timberwolves,2023-24,9,5.7
11,1610612754,Indiana Pacers,2023-24,8,7.5
19,1610612752,New York Knicks,2023-24,7,5.8
7,1610612743,Denver Nuggets,2023-24,7,11.2
20,1610612760,Oklahoma City Thunder,2023-24,6,13.2
5,1610612739,Cleveland Cavaliers,2023-24,5,4.3
21,1610612753,Orlando Magic,2023-24,3,3.1
16,1610612749,Milwaukee Bucks,2023-24,2,5.8


O SVR tem uma abordagem filosófica diferente. Em vez de tentar traçar uma linha perfeita (Regressão Linear) ou criar um labirinto de regras (Árvores), ele tenta criar um "tubo" (margem de tolerância) e colocar o máximo de times dentro desse tubo. Ele foca em punir apenas quem fica muito fora da curva. Isso pode ser excelente para lidar com times que "quase" chegaram lá.

In [18]:
from sklearn.svm import SVR

# Inicializar o SVR
# kernel='rbf': (Radial Basis Function) permite criar fronteiras curvas/não-lineares
# C: Penalidade por errar. Valores maiores forçam o modelo a tentar acertar mais (risco de overfitting)
# epsilon: A largura do "tubo" de tolerância (margem onde ele não penaliza o erro)
svr_pca = SVR(kernel='rbf', C=1.0, epsilon=0.2)

# Treinar o Modelo
svr_pca.fit(X_train, y_train)

# Fazer Previsões
y_pred_svr = svr_pca.predict(X_test)

# Avaliar usando a nossa função padronizada
metricas_svr = avaliar_modelo("SVM - SVR (PCA-5)", y_test, y_pred_svr)

--- Resultados: SVM - SVR (PCA-5) ---
MAE  (Erro médio absoluto):      1.9779
MSE  (Penaliza erros grandes):   11.2052
RMSE (Mesma unidade de y):       3.3474
R²   (Ganho sobre a média):      0.3632


Visualização das previsões:

In [20]:
# Organizar e Visualizar os Resultados
resultados_svr = X_test_full[['TEAM_ID', 'SEASON_ID']].copy()
resultados_svr['TEAM_NAME'] = resultados_svr['TEAM_ID'].map(nba_teams)
resultados_svr['VITORIAS_REAIS'] = y_test.values
resultados_svr['PREVISAO_SVR'] = y_pred_svr.round(1)

# Reorganizar colunas
colunas_ordem = ['TEAM_ID', 'TEAM_NAME', 'SEASON_ID', 'VITORIAS_REAIS', 'PREVISAO_SVR']
resultados_svr = resultados_svr[colunas_ordem]

print("\n--- Ranking dos Times (SVM - SVR) ---")
display(resultados_svr.sort_values(by='VITORIAS_REAIS', ascending=False))


--- Ranking dos Times (SVM - SVR) ---


,TEAM_ID,TEAM_NAME,SEASON_ID,VITORIAS_REAIS,PREVISAO_SVR
1,1610612738,Boston Celtics,2023-24,16,5.2
6,1610612742,Dallas Mavericks,2023-24,13,2.5
17,1610612750,Minnesota Timberwolves,2023-24,9,7.8
11,1610612754,Indiana Pacers,2023-24,8,2.4
19,1610612752,New York Knicks,2023-24,7,4.2
7,1610612743,Denver Nuggets,2023-24,7,4.5
20,1610612760,Oklahoma City Thunder,2023-24,6,5.5
5,1610612739,Cleveland Cavaliers,2023-24,5,3.6
21,1610612753,Orlando Magic,2023-24,3,3.1
16,1610612749,Milwaukee Bucks,2023-24,2,3.2


As Redes Neurais são a base da inteligência artificial moderna. Elas funcionam criando "neurônios" matemáticos organizados em camadas ocultas. A vantagem é que elas podem aprender qualquer função matemática existente. A desvantagem é que elas precisam de muitos dados (frequentemente dezenas de milhares de linhas) para não memorizarem o treino. Nosso dataset tem cerca de 800 linhas.

In [21]:
from sklearn.neural_network import MLPRegressor
import warnings
from sklearn.exceptions import ConvergenceWarning

# Ignorar os avisos chatos se a rede demorar muito para convergir
warnings.filterwarnings('ignore', category=ConvergenceWarning)

# Inicializar a Rede Neural (MLP)
# hidden_layer_sizes=(64, 32): Duas camadas ocultas, a primeira com 64 neurônios e a segunda com 32
# activation='relu': A função de ativação mais usada no mundo para redes profundas
# solver='adam': O otimizador de pesos (padrão da indústria)
# max_iter=500: Número máximo de vezes que a rede vai passar pelos dados para aprender
mlp_pca = MLPRegressor(hidden_layer_sizes=(64, 32), activation='relu', solver='adam', 
                       max_iter=500, random_state=42)

# Treinar o Modelo
mlp_pca.fit(X_train, y_train)

# Fazer Previsões
y_pred_mlp = mlp_pca.predict(X_test)

# Avaliar usando a nossa função padronizada
metricas_mlp = avaliar_modelo("Rede Neural - MLP (PCA-5)", y_test, y_pred_mlp)

--- Resultados: Rede Neural - MLP (PCA-5) ---
MAE  (Erro médio absoluto):      1.9975
MSE  (Penaliza erros grandes):   8.8979
RMSE (Mesma unidade de y):       2.9829
R²   (Ganho sobre a média):      0.4943


Visualização das predições:

In [22]:
# Organizar e Visualizar os Resultados
resultados_mlp = X_test_full[['TEAM_ID', 'SEASON_ID']].copy()
resultados_mlp['TEAM_NAME'] = resultados_mlp['TEAM_ID'].map(nba_teams)
resultados_mlp['VITORIAS_REAIS'] = y_test.values
resultados_mlp['PREVISAO_MLP'] = y_pred_mlp.round(1)

# Reorganizar colunas
colunas_ordem = ['TEAM_ID', 'TEAM_NAME', 'SEASON_ID', 'VITORIAS_REAIS', 'PREVISAO_MLP']
resultados_mlp = resultados_mlp[colunas_ordem]

print("\n--- Top 10 Times (Rede Neural) ---")
display(resultados_mlp.sort_values(by='VITORIAS_REAIS', ascending=False))


--- Top 10 Times (Rede Neural) ---


,TEAM_ID,TEAM_NAME,SEASON_ID,VITORIAS_REAIS,PREVISAO_MLP
1,1610612738,Boston Celtics,2023-24,16,13.8
6,1610612742,Dallas Mavericks,2023-24,13,3.8
17,1610612750,Minnesota Timberwolves,2023-24,9,9.5
11,1610612754,Indiana Pacers,2023-24,8,5.3
19,1610612752,New York Knicks,2023-24,7,5.6
7,1610612743,Denver Nuggets,2023-24,7,8.9
20,1610612760,Oklahoma City Thunder,2023-24,6,9.5
5,1610612739,Cleveland Cavaliers,2023-24,5,5.5
21,1610612753,Orlando Magic,2023-24,3,4.1
16,1610612749,Milwaukee Bucks,2023-24,2,4.9


Apartir de agora, tentaremos otimizar os modelos já implementados aqui, com um grid search em cada um, exceto a regressão linear.

Usamos MAE na otimização porque queremos que o modelo seja corajoso o suficiente para encontrar o campeão de 16 vitórias e ignorar os acidentes de percurso (lesões, zebras).

Para curar o overfitting de uma árvore, além da profundidade, nós adicionamos regras de "poda" (parar de crescer se houverem poucos times naquele galho).

Por isso, na nossa grade (param_grid), vamos incluir também o min_samples_split e o min_samples_leaf.

In [23]:
from sklearn.model_selection import GridSearchCV

print("Iniciando GridSearchCV para a Árvore de Decisão...")

#  Definir o modelo base
dt_base = DecisionTreeRegressor(random_state=42)

# Definir a grade de hiperparâmetros (O "Cardápio" de testes)
# max_depth: Até quantos níveis a árvore pode descer (3 a 7 para não ficar muito complexa)
# min_samples_split: Mínimo de times necessários para permitir que um nó se divida em dois
# min_samples_leaf: Mínimo de times que devem sobrar na ponta final da árvore (folha)
param_grid_dt = {
    'max_depth': [3, 4, 5, 6, 7],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 5, 10, 15]
}

# Configurar o GridSearchCV
# cv=5: Divide o treino em 5 partes para ter certeza que o resultado não é sorte (Cross-Validation)
# scoring: O que ele deve tentar otimizar. Vamos pedir para ele minimizar o MAE.
# n_jobs=-1: Usa todos os núcleos do processador para ir mais rápido
grid_dt = GridSearchCV(estimator=dt_base, 
                       param_grid=param_grid_dt, 
                       scoring='neg_mean_absolute_error', 
                       cv=5, 
                       n_jobs=-1)

# Executar o treinamento de todas as combinações (pode levar alguns segundos)
grid_dt.fit(X_train, y_train)

# Resgatar o melhor modelo encontrado
melhor_dt = grid_dt.best_estimator_

print("\n--- Melhor Configuração Encontrada (Decision Tree) ---")
print(grid_dt.best_params_)

# Avaliar o modelo otimizado usando a nossa função padronizada
y_pred_dt_opt = melhor_dt.predict(X_test)
metricas_dt_opt = avaliar_modelo("Decision Tree Otimizada (PCA-5)", y_test, y_pred_dt_opt)

Iniciando GridSearchCV para a Árvore de Decisão...

--- Melhor Configuração Encontrada (Decision Tree) ---
{'max_depth': 6, 'min_samples_leaf': 15, 'min_samples_split': 2}
--- Resultados: Decision Tree Otimizada (PCA-5) ---
MAE  (Erro médio absoluto):      2.7292
MSE  (Penaliza erros grandes):   14.1603
RMSE (Mesma unidade de y):       3.7630
R²   (Ganho sobre a média):      0.1952


Visualização das previsões:

In [24]:
# Visualizar 
resultados_dt_opt = X_test_full[['TEAM_ID', 'SEASON_ID']].copy()
resultados_dt_opt['TEAM_NAME'] = resultados_dt_opt['TEAM_ID'].map(nba_teams)
resultados_dt_opt['VITORIAS_REAIS'] = y_test.values
resultados_dt_opt['PREVISAO_DT_OPT'] = y_pred_dt_opt.round(1)

# Reorganizar colunas
colunas_ordem = ['TEAM_ID', 'TEAM_NAME', 'SEASON_ID', 'VITORIAS_REAIS', 'PREVISAO_DT_OPT']
resultados_dt_opt = resultados_dt_opt[colunas_ordem]

print("\n--- Top 10 Times (Decision Tree OTIMIZADA) ---")
display(resultados_dt_opt.sort_values(by='VITORIAS_REAIS', ascending=False))


--- Top 10 Times (Decision Tree OTIMIZADA) ---


,TEAM_ID,TEAM_NAME,SEASON_ID,VITORIAS_REAIS,PREVISAO_DT_OPT
1,1610612738,Boston Celtics,2023-24,16,7.1
6,1610612742,Dallas Mavericks,2023-24,13,2.3
17,1610612750,Minnesota Timberwolves,2023-24,9,10.2
11,1610612754,Indiana Pacers,2023-24,8,2.3
19,1610612752,New York Knicks,2023-24,7,7.7
7,1610612743,Denver Nuggets,2023-24,7,10.2
20,1610612760,Oklahoma City Thunder,2023-24,6,10.2
5,1610612739,Cleveland Cavaliers,2023-24,5,5.5
21,1610612753,Orlando Magic,2023-24,3,1.6
16,1610612749,Milwaukee Bucks,2023-24,2,2.3


No nosso teste anterior, o Random Forest original já tinha sido muito bom (MAE de 2.33). O desafio do GridSearch agora será ver se conseguimos abaixar esse erro para perto da marca de 2.0 (o nível do SVR e do MLP).

In [25]:
print("Iniciando GridSearchCV para o Random Forest...")

# Definir o modelo base
rf_base = RandomForestRegressor(random_state=42)

# Definir a grade de hiperparâmetros
# n_estimators: Quantas árvores na floresta. 50, 100 e 200 são valores clássicos.
# max_depth: A profundidade. Deixamos um pouco maior que a DT porque o RF dilui o overfitting na média.
# min_samples_leaf: O mesmo conceito de poda que usamos na DT.
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 7, 10, None], # None permite que as árvores cresçam até o fim (arriscado, mas o RF lida bem)
    'min_samples_leaf': [5, 10, 15], 
    'min_samples_split': [2, 5, 10]
}

# Configurar o GridSearchCV
grid_rf = GridSearchCV(estimator=rf_base, 
                       param_grid=param_grid_rf, 
                       scoring='neg_mean_absolute_error', 
                       cv=5, 
                       n_jobs=-1)

# Executar o treinamento (Isso vai demorar um pouco mais, pois são muitas árvores!)
grid_rf.fit(X_train, y_train)

# Resgatar o melhor modelo encontrado
melhor_rf = grid_rf.best_estimator_

print("\n--- Melhor Configuração Encontrada (Random Forest) ---")
print(grid_rf.best_params_)

# Avaliar o modelo otimizado
y_pred_rf_opt = melhor_rf.predict(X_test)
metricas_rf_opt = avaliar_modelo("Random Forest Otimizado (PCA-5)", y_test, y_pred_rf_opt)


Iniciando GridSearchCV para o Random Forest...

--- Melhor Configuração Encontrada (Random Forest) ---
{'max_depth': 10, 'min_samples_leaf': 10, 'min_samples_split': 2, 'n_estimators': 50}
--- Resultados: Random Forest Otimizado (PCA-5) ---
MAE  (Erro médio absoluto):      2.2223
MSE  (Penaliza erros grandes):   12.6117
RMSE (Mesma unidade de y):       3.5513
R²   (Ganho sobre a média):      0.2832


Visualização das previsões:

In [ ]:
# Visualizar
resultados_rf_opt = X_test_full[['TEAM_ID', 'SEASON_ID']].copy()
resultados_rf_opt['TEAM_NAME'] = resultados_rf_opt['TEAM_ID'].map(nba_teams)
resultados_rf_opt['VITORIAS_REAIS'] = y_test.values
resultados_rf_opt['PREVISAO_RF_OPT'] = y_pred_rf_opt.round(1)

# Reorganizar colunas
colunas_ordem = ['TEAM_ID', 'TEAM_NAME', 'SEASON_ID', 'VITORIAS_REAIS', 'PREVISAO_RF_OPT']
resultados_rf_opt = resultados_rf_opt[colunas_ordem]

print("\n--- Ranking dos Times (Random Forest OTIMIZADO) ---")
display(resultados_rf_opt.sort_values(by='VITORIAS_REAIS', ascending=False))


--- Ranking dos Times (Random Forest OTIMIZADO) ---


,TEAM_ID,TEAM_NAME,SEASON_ID,VITORIAS_REAIS,PREVISAO_RF_OPT
1,1610612738,Boston Celtics,2023-24,16,7.7
6,1610612742,Dallas Mavericks,2023-24,13,2.3
17,1610612750,Minnesota Timberwolves,2023-24,9,9.4
11,1610612754,Indiana Pacers,2023-24,8,2.8
19,1610612752,New York Knicks,2023-24,7,6.2
7,1610612743,Denver Nuggets,2023-24,7,7.5
20,1610612760,Oklahoma City Thunder,2023-24,6,7.3
5,1610612739,Cleveland Cavaliers,2023-24,5,6.1
21,1610612753,Orlando Magic,2023-24,3,2.7
16,1610612749,Milwaukee Bucks,2023-24,2,2.3


O nosso XGBoost original sofreu um overfitting agressivo (MAE de 2.46, R^2 de 0.16).

Para a grade do XGBoost, vamos focar em árvores rasas (max_depth baixo) e testar a taxa de aprendizado (learning_rate). Uma taxa menor faz o modelo aprender mais devagar, mas com muito mais segurança, o que é excelente para evitar o overfitting.

In [28]:
print("Iniciando GridSearchCV para o XGBoost...")

# Definir o modelo base
xgb_base = XGBRegressor(random_state=42)

# Definir a grade de hiperparâmetros
# max_depth: No XGBoost, usamos árvores bem mais rasas (3 a 5) porque ele as constrói em série
# learning_rate: O "tamanho do passo" da correção. Valores menores (0.01, 0.05) evitam overfitting
# n_estimators: Quantas árvores em série vamos construir
param_grid_xgb = {
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [50, 100, 200]
}

# Configurar o GridSearchCV
grid_xgb = GridSearchCV(estimator=xgb_base, 
                       param_grid=param_grid_xgb, 
                       scoring='neg_mean_absolute_error', 
                       cv=5, 
                       n_jobs=-1)

# Executar o treinamento 
grid_xgb.fit(X_train, y_train)

# Resgatar o melhor modelo encontrado
melhor_xgb = grid_xgb.best_estimator_

print("\n--- Melhor Configuração Encontrada (XGBoost) ---")
print(grid_xgb.best_params_)

# Avaliar o modelo otimizado
y_pred_xgb_opt = melhor_xgb.predict(X_test)
metricas_xgb_opt = avaliar_modelo("XGBoost Otimizado (PCA-5)", y_test, y_pred_xgb_opt)

Iniciando GridSearchCV para o XGBoost...

--- Melhor Configuração Encontrada (XGBoost) ---
{'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 100}
--- Resultados: XGBoost Otimizado (PCA-5) ---
MAE  (Erro médio absoluto):      2.3113
MSE  (Penaliza erros grandes):   13.4195
RMSE (Mesma unidade de y):       3.6633
R²   (Ganho sobre a média):      0.2373


Visualização das previsões:

In [29]:
# Visualizar
resultados_xgb_opt = X_test_full[['TEAM_ID', 'SEASON_ID']].copy()
resultados_xgb_opt['TEAM_NAME'] = resultados_xgb_opt['TEAM_ID'].map(nba_teams)
resultados_xgb_opt['VITORIAS_REAIS'] = y_test.values
resultados_xgb_opt['PREVISAO_XGB_OPT'] = y_pred_xgb_opt.round(1)

# Reorganizar colunas
colunas_ordem = ['TEAM_ID', 'TEAM_NAME', 'SEASON_ID', 'VITORIAS_REAIS', 'PREVISAO_XGB_OPT']
resultados_xgb_opt = resultados_xgb_opt[colunas_ordem]

print("\n--- Ranking dos Times (XGBoost OTIMIZADO) ---")
display(resultados_xgb_opt.sort_values(by='VITORIAS_REAIS', ascending=False))


--- Ranking dos Times (XGBoost OTIMIZADO) ---


,TEAM_ID,TEAM_NAME,SEASON_ID,VITORIAS_REAIS,PREVISAO_XGB_OPT
1,1610612738,Boston Celtics,2023-24,16,9.1
6,1610612742,Dallas Mavericks,2023-24,13,2.7
17,1610612750,Minnesota Timberwolves,2023-24,9,7.8
11,1610612754,Indiana Pacers,2023-24,8,5.0
19,1610612752,New York Knicks,2023-24,7,4.7
7,1610612743,Denver Nuggets,2023-24,7,7.9
20,1610612760,Oklahoma City Thunder,2023-24,6,11.7
5,1610612739,Cleveland Cavaliers,2023-24,5,4.6
21,1610612753,Orlando Magic,2023-24,3,1.7
16,1610612749,Milwaukee Bucks,2023-24,2,3.1


O SVR teve o segundo melhor MAE do projeto (1.97), sendo conservador na quantidade de vitórias. O GridSearch agora vai tentar achar o equilíbrio perfeito entre não ser tão conservador (ele deu só 5.2 vitórias pro Celtics), sem perder a precisão nos times menores.

In [30]:
print("Iniciando GridSearchCV para o SVR...")

# Definir o modelo base
svr_base = SVR()

# Definir a grade de hiperparâmetros
param_grid_svr = {
    'kernel': ['linear', 'rbf'],
    'C': [0.1, 1.0, 5.0, 10.0],
    'epsilon': [0.05, 0.1, 0.2, 0.5]
}

# Configurar o GridSearchCV
grid_svr = GridSearchCV(estimator=svr_base, 
                       param_grid=param_grid_svr, 
                       scoring='neg_mean_absolute_error', 
                       cv=5, 
                       n_jobs=-1)

# Executar o treinamento 
grid_svr.fit(X_train, y_train)

# Resgatar o melhor modelo encontrado
melhor_svr = grid_svr.best_estimator_

print("\n--- Melhor Configuração Encontrada (SVR) ---")
print(grid_svr.best_params_)

# Avaliar o modelo otimizado
y_pred_svr_opt = melhor_svr.predict(X_test)
metricas_svr_opt = avaliar_modelo("SVM - SVR Otimizado (PCA-5)", y_test, y_pred_svr_opt)

Iniciando GridSearchCV para o SVR...

--- Melhor Configuração Encontrada (SVR) ---
{'C': 5.0, 'epsilon': 0.05, 'kernel': 'rbf'}
--- Resultados: SVM - SVR Otimizado (PCA-5) ---
MAE  (Erro médio absoluto):      1.9213
MSE  (Penaliza erros grandes):   10.4104
RMSE (Mesma unidade de y):       3.2265
R²   (Ganho sobre a média):      0.4083


Visualização das previsões:

In [31]:
# Visualizar 
resultados_svr_opt = X_test_full[['TEAM_ID', 'SEASON_ID']].copy()
resultados_svr_opt['TEAM_NAME'] = resultados_svr_opt['TEAM_ID'].map(nba_teams)
resultados_svr_opt['VITORIAS_REAIS'] = y_test.values
resultados_svr_opt['PREVISAO_SVR_OPT'] = y_pred_svr_opt.round(1)

# Reorganizar colunas
colunas_ordem = ['TEAM_ID', 'TEAM_NAME', 'SEASON_ID', 'VITORIAS_REAIS', 'PREVISAO_SVR_OPT']
resultados_svr_opt = resultados_svr_opt[colunas_ordem]

print("\n--- Ranking dos Times (SVR OTIMIZADO) ---")
display(resultados_svr_opt.sort_values(by='VITORIAS_REAIS', ascending=False))


--- Ranking dos Times (SVR OTIMIZADO) ---


,TEAM_ID,TEAM_NAME,SEASON_ID,VITORIAS_REAIS,PREVISAO_SVR_OPT
1,1610612738,Boston Celtics,2023-24,16,8.2
6,1610612742,Dallas Mavericks,2023-24,13,1.7
17,1610612750,Minnesota Timberwolves,2023-24,9,10.6
11,1610612754,Indiana Pacers,2023-24,8,2.7
19,1610612752,New York Knicks,2023-24,7,5.0
7,1610612743,Denver Nuggets,2023-24,7,7.4
20,1610612760,Oklahoma City Thunder,2023-24,6,8.4
5,1610612739,Cleveland Cavaliers,2023-24,5,5.0
21,1610612753,Orlando Magic,2023-24,3,2.7
16,1610612749,Milwaukee Bucks,2023-24,2,3.1


A nossa Rede Neural original já era a melhor (MAE de 1.99 e R^2 de 0.49). O perigo agora é que, ao tentar otimizar, a gente a force ao overfitting.

In [32]:
warnings.filterwarnings('ignore', category=ConvergenceWarning)

print("Iniciando GridSearchCV para a Rede Neural (MLP)...")

# Definir o modelo base
mlp_base = MLPRegressor(max_iter=500, random_state=42)

# Definir a grade de hiperparâmetros
param_grid_mlp = {
    'hidden_layer_sizes': [(64,), (64, 32), (32, 16)], 
    'activation': ['relu', 'tanh'],
    'solver': ['adam', 'lbfgs'] # lbfgs é excelente para bases de dados menores como a nossa
}

# Configurar o GridSearchCV
grid_mlp = GridSearchCV(estimator=mlp_base, 
                       param_grid=param_grid_mlp, 
                       scoring='neg_mean_absolute_error', 
                       cv=5, 
                       n_jobs=-1)

# Executar o treinamento
grid_mlp.fit(X_train, y_train)

# Resgatar o melhor modelo encontrado
melhor_mlp = grid_mlp.best_estimator_

print("\n--- Melhor Configuração Encontrada (MLP) ---")
print(grid_mlp.best_params_)

# Avaliar o modelo otimizado
y_pred_mlp_opt = melhor_mlp.predict(X_test)
metricas_mlp_opt = avaliar_modelo("Rede Neural Otimizada (PCA-5)", y_test, y_pred_mlp_opt)

Iniciando GridSearchCV para a Rede Neural (MLP)...

--- Melhor Configuração Encontrada (MLP) ---
{'activation': 'relu', 'hidden_layer_sizes': (32, 16), 'solver': 'adam'}
--- Resultados: Rede Neural Otimizada (PCA-5) ---
MAE  (Erro médio absoluto):      2.0284
MSE  (Penaliza erros grandes):   8.9400
RMSE (Mesma unidade de y):       2.9900
R²   (Ganho sobre a média):      0.4919


Visualização das previsões:

In [33]:
# Visualizar 
resultados_mlp_opt = X_test_full[['TEAM_ID', 'SEASON_ID']].copy()
resultados_mlp_opt['TEAM_NAME'] = resultados_mlp_opt['TEAM_ID'].map(nba_teams)
resultados_mlp_opt['VITORIAS_REAIS'] = y_test.values
resultados_mlp_opt['PREVISAO_MLP_OPT'] = y_pred_mlp_opt.round(1)

# Reorganizar colunas
colunas_ordem = ['TEAM_ID', 'TEAM_NAME', 'SEASON_ID', 'VITORIAS_REAIS', 'PREVISAO_MLP_OPT']
resultados_mlp_opt = resultados_mlp_opt[colunas_ordem]

print("\n--- Ranking dos Times (MLP OTIMIZADO) ---")
display(resultados_mlp_opt.sort_values(by='VITORIAS_REAIS', ascending=False))


--- Ranking dos Times (MLP OTIMIZADO) ---


,TEAM_ID,TEAM_NAME,SEASON_ID,VITORIAS_REAIS,PREVISAO_MLP_OPT
1,1610612738,Boston Celtics,2023-24,16,14.6
6,1610612742,Dallas Mavericks,2023-24,13,3.4
17,1610612750,Minnesota Timberwolves,2023-24,9,10.1
11,1610612754,Indiana Pacers,2023-24,8,3.8
19,1610612752,New York Knicks,2023-24,7,6.2
7,1610612743,Denver Nuggets,2023-24,7,8.2
20,1610612760,Oklahoma City Thunder,2023-24,6,9.4
5,1610612739,Cleveland Cavaliers,2023-24,5,4.9
21,1610612753,Orlando Magic,2023-24,3,5.0
16,1610612749,Milwaukee Bucks,2023-24,2,5.0


Apesar de o MLP otimizado ter aumentado o MAE, ele fez uma previsão melhor do time campeão, e isso tem mais valor do que um aumento dos erros dos times ruins/medianos.

### Resultados da Regressão e Otimização

| Modelo e Hiperparâmetros | MAE (Erro Absoluto) | MSE | RMSE | R² (Variância) | Previsão Campeão (Boston: 16) |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Regressão Linear** | 2.6231 | 10.9812 | 3.3138 | 0.3759 | 8.7 |
| **Decision Tree**<br>`max_depth=6, min_samples_leaf=15, min_samples_split=2` | 2.7292 | 14.1603 | 3.7630 | 0.1952 | 7.1 |
| **Random Forest**<br>`max_depth=10, min_samples_leaf=10, min_samples_split=2, n_estimators=50` | 2.2223 | 12.6117 | 3.5513 | 0.2832 | 7.7 |
| **XGBoost**<br>`learning_rate=0.05, max_depth=5, n_estimators=100` | 2.3113 | 13.4195 | 3.6633 | 0.2373 | 9.1 |
| **SVR**<br>`C=5.0, epsilon=0.05, kernel=rbf` | **1.9213**⭐ | 10.4104 | 3.2265 | 0.4083 | 8.2 |
| **Rede Neural**<br>`activation=relu, hidden_layer_sizes=(32, 16), solver=adam` | 2.0284 | **8.9400**⭐ | **2.9900**⭐ | **0.4919**⭐ | **14.6**⭐ |